# 03 · Score & rank villages

**Purpose:** combine the satellite signals into a growth score and select the **top 100 villages** – overall and "rural" (away from big cities).

| | |
|---|---|
| **Inputs** (`data/processed/`) | `villages_india.gpkg` (01) · `lights_built_india_*.csv` (02) · `buildings_shortlist.csv` (Stage 2, this notebook) · pilot files for the appendix |
| **Inputs** (`data/raw/`) | `village_modified.gpkg` (city boundaries) · `shrid_pop_area_key.csv` (city populations) |
| **Outputs** | `data/processed/shortlist_ids.csv` · **`outputs/top100_villages.csv`** · **`outputs/top100_rural_villages.csv`** |
| **Run time** | ~5 min (loading 590k villages + distance to cities) |

**Pipeline position:** 01 Prepare → 02 Extract → **03 Score & rank** → 04 Visualise

### Two-stage design
1. **Stage 1 – screen all of India** (591k villages) with the two signals that are affordable nationally: night lights and built-up share → shortlist the best candidates.
2. **Stage 2 – confirm the shortlist** with an independent signal (Open Buildings building counts) → final top 100.

This keeps Earth Engine compute within the free quota while still cross-checking every final village with a second, independent data source.

In [16]:
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

## Method decisions

All tunable choices live in one cell so they are easy to find, justify and change.

>  **My reasoning for each choice:** 
> - **`MIN_POP = 500`** – 123 Haryana villages had under 100 people; 401,430 villages nationally have 500+.
> - **`WEIGHTS`** – Night lights and built-up area are weighted equally because there is no village-level income data to test which one better predicts growth, and the two capture different sides of it: activity (lights) and physical expansion (built-up), with a correlation of only 0.39. Giving them equal weight keeps either signal’s weaknesses from dominating the score.
> - **`REQUIRE_BUILT_GROWTH` + light-only flag** – 447 villages flagged. Disang Kinar Bangali: lights 1.2 → 125 with built-up flat at 4%. Konsari: lights 0.4 → 22 with built-up 6% → 24%.
> - **`RURAL_KM = 10`** 61 of the overall top 100 are within 10 km of a city (median 6.7 km). The rural list’s median is 18.7 km. 39 villages appear on both lists.
> - **`W_BLDG = 0.3`** – how much the independent buildings check should count


In [17]:
MIN_POP = 500                 # ignore very small villages (tiny base -> unstable growth rates)
WEIGHTS = {"ntl_growth": 0.5,  # Stage 1 weight of night-light growth
           "built_growth": 0.5}  # Stage 1 weight of built-up growth
REQUIRE_BUILT_GROWTH = True   # a village must show SOME physical growth to qualify
FLAG_QUANTILE = 0.99          # "light-only spike" = top 1% light growth with no built-up growth
CITY_POP = 100_000            # a "city" = town with 100k+ people in 2011
RURAL_KM = 10                 # rural list = villages more than this far from a city
SHORTLIST = 250               # candidates per list sent to the Stage 2 buildings check
W_BLDG = 0.3                  # weight of building growth in the final score
TOP_N = 100

RUN_STAGE2_EXPORT = False     # True = re-run the Earth Engine buildings export (uses compute quota)

## 1 · Load villages and Stage 1 signals

In [18]:
files = sorted(PROC.glob("lights_built_india_*.csv"))
lb = pd.concat([pd.read_csv(f, dtype={"shrid2": str}) for f in files])
# Drop Earth Engine bookkeeping columns and columns already in the village file
lb = lb.drop(columns=["system:index", ".geo", "pop_2011", "state"], errors="ignore")

villages = gpd.read_file(PROC / "villages_india.gpkg")
df = villages.merge(lb, on="shrid2")
print(f"{len(files)} state files | {len(df):,} villages with satellite data")

33 state files | 591,717 villages with satellite data


## 2 · Growth features

- **Night-light growth** = `log(1 + lights 2025) − log(1 + lights 2020)` – a growth *rate* that is not exaggerated by near-zero starting values.
- **Built-up growth** = change in the built-up *share* of the village (percentage points).

In [19]:
df["ntl_growth"] = np.log1p(df["ntl_2025"]) - np.log1p(df["ntl_2020"])
df["built_growth"] = df["built_2025"] - df["built_2020"]

# A handful of villages (mountains, tiny islands) have no satellite value -> drop them
before = len(df)
df = df.dropna(subset=["ntl_growth", "built_growth"]).copy()
print(f"Dropped {before - len(df)} villages with missing values -> {len(df):,} remain")

# Census 2011 state codes -> names (Telangana was still part of Andhra Pradesh in 2011)
STATES = {"01": "Jammu & Kashmir", "02": "Himachal Pradesh", "03": "Punjab", "04": "Chandigarh",
          "05": "Uttarakhand", "06": "Haryana", "07": "Delhi", "08": "Rajasthan", "09": "Uttar Pradesh",
          "10": "Bihar", "11": "Sikkim", "12": "Arunachal Pradesh", "13": "Nagaland", "14": "Manipur",
          "15": "Mizoram", "16": "Tripura", "17": "Meghalaya", "18": "Assam", "19": "West Bengal",
          "20": "Jharkhand", "21": "Odisha", "22": "Chhattisgarh", "23": "Madhya Pradesh", "24": "Gujarat",
          "25": "Daman & Diu", "26": "Dadra & Nagar Haveli", "27": "Maharashtra",
          "28": "Andhra Pradesh (incl. Telangana)", "29": "Karnataka", "30": "Goa", "31": "Lakshadweep",
          "32": "Kerala", "33": "Tamil Nadu", "34": "Puducherry", "35": "Andaman & Nicobar"}
df["state_name"] = df["pc11_state_id"].map(STATES)

df[["ntl_growth", "built_growth"]].describe().round(3)

Dropped 130 villages with missing values -> 591,587 remain


,ntl_growth,built_growth
count,591587.000,591587.000
mean,0.166,0.011
std,0.158,0.037
min,-3.289,-0.647
25%,0.074,-0.001
50%,0.125,0.004
75%,0.220,0.018
max,4.056,0.975


## 3 · Distance to the nearest city

Fast growth next to a city is often the city spreading outwards. To tell this apart from villages growing on their own, every village gets the distance (km) from its centre to the edge of the nearest town with 100,000+ people.

In [20]:
def load_cities(min_pop=CITY_POP):
    # Statutory towns have PC11 codes >= 800000; keep those above the population threshold
    towns = gpd.read_file(RAW / "village_modified.gpkg", where="pc11_town_village_id >= '800000'")
    towns["shrid2"] = ("11-" + towns["pc11_state_id"] + "-" + towns["pc11_district_id"] + "-"
                       + towns["pc11_subdistrict_id"] + "-" + towns["pc11_town_village_id"])
    tpop = pd.read_csv(RAW / "shrid_pop_area_key.csv", dtype={"shrid2": str},
                       usecols=["shrid2", "shrid_pc11_pca_tot_p"])
    cities = towns.merge(tpop, on="shrid2")
    cities = cities[cities["shrid_pc11_pca_tot_p"] >= min_pop]
    return cities[["town_village_name", "geometry"]].rename(columns={"town_village_name": "nearest_city"})


def add_city_distance(gdf, cities, round_km=1):
    # Nearest-city join in an India-wide projection (EPSG:7755, metres)
    pts = gdf[["geometry"]].copy()
    pts["geometry"] = pts.geometry.representative_point()       # a point guaranteed inside the village
    near = gpd.sjoin_nearest(pts.to_crs(7755), cities.to_crs(7755), distance_col="dist_m")
    near = near[~near.index.duplicated()]                       # ties: keep one city
    gdf["nearest_city"] = near["nearest_city"]
    km = near["dist_m"] / 1000
    gdf["dist_city_km"] = km.round(round_km) if round_km is not None else km
    return gdf


cities = load_cities()
print(f"Cities with {CITY_POP:,}+ people: {len(cities)}")
df = add_city_distance(df, cities)
df["dist_city_km"].describe().round(1)

Cities with 100,000+ people: 447


count    591587.0
mean         39.5
std          30.0
min           0.0
25%          18.7
50%          32.5
75%          52.2
max         557.7
Name: dist_city_km, dtype: float64

## 4 · Stage 1 score and shortlists

1. Keep villages with at least `MIN_POP` people.
2. **Flag light-only spikes** – top 1% light growth but no built-up growth → likely gas flares or a single facility, not village growth.
3. Convert each signal to a **national percentile** (0–100) so lights and built-up are on the same scale and outliers can't dominate.
4. **Stage 1 score** = weighted average of the percentiles.
5. Shortlist the best `SHORTLIST` villages overall, and the best `SHORTLIST` more than `RURAL_KM` from a city.

In [21]:
s = df[df["pop_2011"] >= MIN_POP].copy()

s["flag_light_only"] = ((s["ntl_growth"] > s["ntl_growth"].quantile(FLAG_QUANTILE))
                        & (s["built_growth"] <= 0))

for col in WEIGHTS:
    s[col + "_pct"] = s[col].rank(pct=True) * 100
s["score_stage1"] = sum(s[c + "_pct"] * w for c, w in WEIGHTS.items())

eligible = s[s["built_growth"] > 0] if REQUIRE_BUILT_GROWTH else s
eligible = eligible[~eligible["flag_light_only"]]

short_all = eligible.nlargest(SHORTLIST, "score_stage1")
short_rural = eligible[eligible["dist_city_km"] > RURAL_KM].nlargest(SHORTLIST, "score_stage1")

shortlist_ids = sorted(set(short_all["shrid2"]) | set(short_rural["shrid2"]))
pd.Series(shortlist_ids, name="shrid2").to_csv(PROC / "shortlist_ids.csv", index=False)

print(f"Villages scored (pop >= {MIN_POP}): {len(s):,}")
print(f"Flagged as light-only spikes: {s['flag_light_only'].sum():,}")
print(f"Eligible: {len(eligible):,}")
print(f"Shortlist: {len(short_all)} overall + {len(short_rural)} rural -> {len(shortlist_ids)} unique villages")

Villages scored (pop >= 500): 401,430
Flagged as light-only spikes: 447
Eligible: 286,317
Shortlist: 250 overall + 250 rural -> 388 unique villages


### Why night lights alone can't be trusted

Two villages with huge light jumps. Only one also shows construction – the other is the kind of case the light-only flag removes.

In [22]:
examples = {"11-18-310-02069-292407": "light spike, no construction (likely gas flare, Assam oil belt)",
            "11-27-508-04058-539319": "light AND construction growth (real expansion)"}
ex = s[s["shrid2"].isin(examples)].copy()
ex["interpretation"] = ex["shrid2"].map(examples)
ex[["town_village_name", "state_name", "ntl_2020", "ntl_2025", "built_2020", "built_2025",
    "flag_light_only", "interpretation"]].round(2)

,town_village_name,state_name,ntl_2020,ntl_2025,built_2020,built_2025,flag_light_only,interpretation
492086,Konsari,Maharashtra,0.44,21.59,0.06,0.24,False,light AND construction growth (real expansion)


## 5 · Stage 2 – buildings check on the shortlist (Earth Engine)

Exports Open Buildings counts (2018 → 2023) for the shortlisted villages only. Switched off by default because `buildings_shortlist.csv` already exists; set `RUN_STAGE2_EXPORT = True` to re-run, then download the CSV from Google Drive into `data/processed/`.

In [23]:
if RUN_STAGE2_EXPORT:
    import ee
    ee.Initialize(project="stately-sentry-472119-i4")
    india_fc = ee.FeatureCollection("projects/stately-sentry-472119-i4/assets/villages_india_shp")
    short_fc = india_fc.filter(ee.Filter.inList("shrid2", shortlist_ids))
    region = ee.Geometry.Rectangle([68, 6, 98, 37])

    def buildings(year):
        # Same definition as notebook 02: mosaic of that year's tiles, building_fractional_count band
        return (ee.ImageCollection("GOOGLE/Research/open-buildings-temporal/v1")
                .filterBounds(region).filterDate(f"{year}-01-01", f"{year + 1}-01-01")
                .select("building_fractional_count").mosaic().rename(f"bldg_{year}"))

    stats = (ee.Image.cat([buildings(2018), buildings(2023)])
             .reduceRegions(collection=short_fc, reducer=ee.Reducer.sum(), scale=4, tileScale=8))
    task = ee.batch.Export.table.toDrive(collection=stats.select([".*"], None, False),
                                         description="buildings_shortlist", folder="village_growth",
                                         fileNamePrefix="buildings_shortlist", fileFormat="CSV")
    task.start()
    print("Stage 2 export started – check with task.status()['state']")
else:
    print("Stage 2 export skipped – using existing buildings_shortlist.csv")

Stage 2 export skipped – using existing buildings_shortlist.csv


## 6 · Final ranking

- **Building counts:** the rasters are stored at 0.5 m but were summed at 4 m, so each value is an 8×8 average → multiply by 64 for real counts.
- **Building growth** = `log(1 + buildings 2023) − log(1 + buildings 2018)`, converted to a percentile *within each shortlist*.
- **Final score** = `(1 − W_BLDG) × Stage 1 score + W_BLDG × building-growth percentile`.

In [24]:
bd = pd.read_csv(PROC / "buildings_shortlist.csv", dtype={"shrid2": str})
bd[["bldg_2018", "bldg_2023"]] *= 64

# Safety check: the buildings file must belong to this exact shortlist
missing = set(shortlist_ids) - set(bd["shrid2"])
assert not missing, f"{len(missing)} shortlisted villages missing from buildings_shortlist.csv - re-run Stage 2"

cand = s.merge(bd[["shrid2", "bldg_2018", "bldg_2023"]], on="shrid2")
cand["bldg_growth"] = np.log1p(cand["bldg_2023"]) - np.log1p(cand["bldg_2018"])


def final_rank(c, n=TOP_N):
    c = c.copy()
    c["bldg_pct"] = c["bldg_growth"].rank(pct=True) * 100
    c["final_score"] = ((1 - W_BLDG) * c["score_stage1"] + W_BLDG * c["bldg_pct"]).round(2)
    c = c.nlargest(n, "final_score")
    c["rank"] = range(1, n + 1)
    return c


top100 = final_rank(cand[cand["shrid2"].isin(short_all["shrid2"])])
top100_rural = final_rank(cand[cand["shrid2"].isin(short_rural["shrid2"])])
top100[["rank", "town_village_name", "state_name", "final_score", "nearest_city", "dist_city_km"]].head(15)

,rank,town_village_name,state_name,final_score,nearest_city,dist_city_km
163,1,Hajishahpur,Bihar,99.87,Bhagalpur,1.5
23,2,Kundal (241),Haryana,99.70,Bahadurgarh,12.8
121,3,Kashroul,Uttar Pradesh,99.69,Gorakhpur,16.6
90,4,Hora Kachhar,Uttar Pradesh,99.40,Kanpur,0.5
55,5,Sajiyali Roopji (Rajaberi),Rajasthan,99.26,Jodhpur,80.1
65,6,Islamabad Kalda,Uttar Pradesh,99.25,Ghaziabad,2.5
5,7,Utar Pora Pahali Pora,Jammu & Kashmir,99.03,Anantnag,23.5
77,8,Banwari Bara,Uttar Pradesh,99.03,Khurja,23.3
230,9,Mundana,Madhya Pradesh,98.98,Pithampur,3.0
9,10,Bounker (39),Punjab,98.60,Ludhiana,7.1


In [25]:
top100_rural[["rank", "town_village_name", "state_name", "final_score", "nearest_city", "dist_city_km"]].head(15)

,rank,town_village_name,state_name,final_score,nearest_city,dist_city_km
23,1,Kundal (241),Haryana,99.82,Bahadurgarh,12.8
121,2,Kashroul,Uttar Pradesh,99.81,Gorakhpur,16.6
55,3,Sajiyali Roopji (Rajaberi),Rajasthan,99.62,Jodhpur,80.1
77,4,Banwari Bara,Uttar Pradesh,99.51,Khurja,23.3
5,5,Utar Pora Pahali Pora,Jammu & Kashmir,99.39,Anantnag,23.5
81,6,Nabipur,Uttar Pradesh,99.14,Palwal,33.7
248,7,Sayakha,Gujarat,99.08,Bharuch,16.5
278,8,Imamguda,Andhra Pradesh (incl. Telangana),98.97,GHMC,14.6
279,9,Athivaram,Andhra Pradesh (incl. Telangana),98.97,Tirupati,44.7
132,10,Sewrahi,Uttar Pradesh,98.71,Bettiah,26.4


## 7 · Validation and key numbers

Figures used in the presentation.

In [26]:
# Does the independent buildings signal confirm the Stage 1 shortlist?
up = (cand["bldg_2023"] > cand["bldg_2018"]).mean()
print(f"Shortlist ({len(cand)} villages): {up:.0%} gained buildings | median buildings "
      f"{cand['bldg_2018'].median():.0f} (2018) -> {cand['bldg_2023'].median():.0f} (2023)")

# How peri-urban are the two lists?
for name, t in [("Overall", top100), ("Rural", top100_rural)]:
    print(f"{name:8s} top {TOP_N}: median distance to city {t['dist_city_km'].median():.1f} km | "
          f"within {RURAL_KM} km: {(t['dist_city_km'] <= RURAL_KM).sum()} | "
          f"top states: {', '.join(t['state_name'].value_counts().head(3).index)}")
print("Villages on both lists:", len(set(top100["shrid2"]) & set(top100_rural["shrid2"])))

Shortlist (388 villages): 99% gained buildings | median buildings 533 (2018) -> 819 (2023)
Overall  top 100: median distance to city 6.7 km | within 10 km: 61 | top states: Karnataka, Uttar Pradesh, Madhya Pradesh
Rural    top 100: median distance to city 18.7 km | within 10 km: 0 | top states: Uttar Pradesh, Andhra Pradesh (incl. Telangana), Rajasthan
Villages on both lists: 39


## 8 · Save the deliverables

In [27]:
OUTPUT_COLS = ["rank", "shrid2", "town_village_name", "state_name", "pop_2011", "final_score",
               "ntl_2020", "ntl_2025", "built_2020", "built_2025", "bldg_2018", "bldg_2023",
               "nearest_city", "dist_city_km", "lat", "lon"]


def save_table(t, name):
    t = t.copy()
    pt = t.geometry.representative_point()           # map location of each village
    t["lat"], t["lon"] = pt.y.round(5), pt.x.round(5)
    t[["bldg_2018", "bldg_2023"]] = t[["bldg_2018", "bldg_2023"]].round(0)
    t[OUTPUT_COLS].round(3).to_csv(OUT / name, index=False)
    print("Saved", OUT / name)


save_table(top100, "top100_villages.csv")
save_table(top100_rural, "top100_rural_villages.csv")

Saved C:\Users\arpit\Downloads\village-growth-intelligence\village-growth-intelligence\outputs\top100_villages.csv
Saved C:\Users\arpit\Downloads\village-growth-intelligence\village-growth-intelligence\outputs\top100_rural_villages.csv


## Appendix · Haryana pilot validation

The pilot (notebook 02, Part A) extracted **all three** signals for every Haryana village at full resolution. Because building counts exist for *every* village there, it tests whether a ranking built only from lights + built-up (Stage 1) also picks villages with strong building growth – i.e. whether Stage 1 is a good screen.

In [28]:
pv = gpd.read_file(PROC / "villages_06.gpkg")
plb = pd.read_csv(PROC / "lights_built_06.csv", dtype={"shrid2": str})
pbd = pd.read_csv(PROC / "buildings_06.csv", dtype={"shrid2": str})
pbd[["bldg_2018", "bldg_2023"]] *= 64                    # same 0.5 m -> 4 m correction

drop = ["system:index", ".geo", "pop_2011"]
p = (pv.merge(plb.drop(columns=drop), on="shrid2")
       .merge(pbd.drop(columns=drop), on="shrid2"))
p["ntl_growth"] = np.log1p(p["ntl_2025"]) - np.log1p(p["ntl_2020"])
p["built_growth"] = p["built_2025"] - p["built_2020"]
p["bldg_growth"] = np.log1p(p["bldg_2023"]) - np.log1p(p["bldg_2018"])
p = add_city_distance(p, cities, round_km=None)

ps = p[p["pop_2011"] >= MIN_POP].copy()
for col in WEIGHTS:
    ps[col + "_pct"] = ps[col].rank(pct=True) * 100
ps["score"] = sum(ps[c + "_pct"] * w for c, w in WEIGHTS.items())
ps["bldg_growth_pct"] = ps["bldg_growth"].rank(pct=True) * 100   # NOT used in the score - only to validate

pilot_top = ps.nlargest(100, "score")
print(f"Pilot villages scored: {len(ps):,}")
print(f"Pilot top 100 - median building-growth percentile: {pilot_top['bldg_growth_pct'].median():.1f}  (50 = no better than random)")
print(f"Pilot top 100 - share within 10 km of a 100k+ city: {(pilot_top['dist_city_km'] < 10).mean():.2f}")
print("Spearman correlation between the three signals:")
ps[["ntl_growth", "built_growth", "bldg_growth"]].corr(method="spearman").round(2)

Pilot villages scored: 6,017
Pilot top 100 - median building-growth percentile: 89.8  (50 = no better than random)
Pilot top 100 - share within 10 km of a 100k+ city: 0.69
Spearman correlation between the three signals:


,ntl_growth,built_growth,bldg_growth
ntl_growth,1.00,0.39,0.06
built_growth,0.39,1.00,0.30
bldg_growth,0.06,0.30,1.00


> **What the pilot tells us:** Top 100 at the 89.8 percentile on building growth (50 = random). Lights ↔ buildings correlation 0.06; built-up ↔ lights 0.39.